<h1>Manipulate Datasets</h1>



This notebook offers simple routines for ad-hoc manipulation of training/verification data

***

<h2>5-year Verif</h2>

My test set for DLTM is 2020 through 2024. Here I create a subset of my land train/test set that only contains these years. This allow convienient comparison of corresponding climatologies without changing my workflow. Note that once the climos have been calculated I can delete this file to free up storage space. The cached climos will be referenced directly and bypass the need for keeping this subset around. 

In [6]:
import xarray as xr 
import dask
from dask.diagnostics import ProgressBar
dask.config.set(num_workers=4); # Processes 4 at a time. Necessary with Perlmutter login memory limits

In [7]:
# Source file
ds_src = xr.open_zarr('/pscratch/sd/n/nacc/training_data/hpx64_2000-2025_dltm-v4_rechunked-time768.zarr')
# Select subset
ds_target = ds_src.sel(time=slice('2020-01-01', '2024-12-31'))
# Wipe encoding AND unify chunks
for var in ds_target.variables:
    ds_target[var].encoding = {}
ds_target = ds_target.chunk({
    'time': 768,
    'channel_c' : 11,
    'channel_in' : 1,
    'channel_out' : 1,
    'face' : 12,
    'height' : 64,
    'width' : 64})
with ProgressBar():
    ds_target.to_zarr('/pscratch/sd/n/nacc/training_data/hpx64_2020-2025_dltm-v4_rechunked-time768.zarr', mode='w')
print(f'Saved dataset successfully.')

[########################################] | 100% Completed | 124.58 s
Saved dataset successfully.


In [4]:
ds_target

<xarray.Dataset> Size: 69GB
Dimensions:      (channel_c: 11, channel_in: 19, channel_out: 5, face: 12,
                  height: 64, width: 64, time: 14616)
Coordinates:
  * channel_c    (channel_c) <U25 1kB 'lsm' 'z' ... 'stl1_annual_range'
  * channel_in   (channel_in) <U14 1kB 'NDVI_gapfill' ... 'tp6-lt1e-8-96H'
  * channel_out  (channel_out) <U12 240B 'NDVI_gapfill' 'swvl1' ... 'stl4'
  * face         (face) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * height       (height) int64 512B 0 1 2 3 4 5 6 7 ... 56 57 58 59 60 61 62 63
    lat          (face, height, width) float64 393kB dask.array<chunksize=(12, 64, 64), meta=np.ndarray>
    lon          (face, height, width) float64 393kB dask.array<chunksize=(12, 64, 64), meta=np.ndarray>
  * time         (time) datetime64[ns] 117kB 2020-01-01 ... 2024-12-31T21:00:00
  * width        (width) int64 512B 0 1 2 3 4 5 6 7 ... 56 57 58 59 60 61 62 63
Data variables:
    constants    (channel_c, face, height, width) float32 2MB dask.array<chunksize=(11, 12, 64, 64), meta=np.ndarray>
    inputs       (time, channel_in, face, height, width) float32 55GB dask.array<chunksize=(768, 1, 12, 64, 64), meta=np.ndarray>
    targets      (time, channel_out, face, height, width) float32 14GB dask.array<chunksize=(768, 1, 12, 64, 64), meta=np.ndarray>